In [2]:
def get_prompt_for_recipe() -> str:
    return """
You are a food assistant.

Based on the user's input in natural language, infer their level of preference for the following ingredient categories.

Available categories:
- oil_fat  
- dairy  
- fruit  
- carbohydrate  
- sweetener  
- processed  
- legume  
- vegetable  
- leafy_vegetable  
- seafood  
- high_protein  
- herb_spice  
- meat

Instructions:
- Carefully analyze the user's statement.
- For each category, assign a score from **-5 to +5**:
  - **+5** means the user strongly prefers this category.
  - **0** means neutral or no clear preference.
  - **-5** means the user strongly avoids this category.
- Provide a JSON-style response containing:
  1. The original user input
  2. A dictionary of category scores (only include relevant categories, or return all with scores)
  3. A brief explanation of your reasoning

Format your response like this:

{
  "user_input": "...",
  "category_scores": {
    "fruit": 2,
    "meat": -3,
    ...
  },
  "reasoning": "..."
}

"""

In [3]:
from openai import OpenAI

client = OpenAI(api_key="sk-bbddcdd5af31410cbe8fd1c76fd64243", base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": get_prompt_for_recipe()},
        {"role": "user", "content": "I want MEAT! AS MORE AS POSSIBLE!"},
    ],
    stream=False
)

print(response.choices[0].message.content)
print(type(response.choices[0].message.content))

{
  "user_input": "I want MEAT! AS MORE AS POSSIBLE!",
  "category_scores": {
    "meat": 5,
    "high_protein": 5
  },
  "reasoning": "The user's statement strongly emphasizes a desire for meat, indicating a very high preference for the 'meat' category. Since meat is also a primary source of high protein, the 'high_protein' category is also scored highly. Other categories are not mentioned, so they are not included in the response."
}
<class 'str'>


In [4]:
import json

content_str = response.choices[0].message.content

parsed = json.loads(content_str)

category_scores = parsed["category_scores"]

print(category_scores)


{'meat': 5, 'high_protein': 5}


In [ ]:
import pandas as pd

df = pd.read_csv("../../MCP/data/diabetic_recipes_with_categories.csv")

label_cols = list(category_scores.keys())

def compute_weighted_score(row):
    score = 0
    for label in label_cols:
        score += row[label] * category_scores.get(label, 0)
    return score

df["weighted_score"] = df.apply(compute_weighted_score, axis=1)

is_diabetic_user = 1

if is_diabetic_user:
    sorted_df = df.sort_values(by="weighted_score", ascending=False)

    top_friendly = sorted_df[sorted_df["is_diabetic_friendly"] == 1].head(3).copy()
    top_friendly["diabetic_note"] = "suitable"

    top_unfriendly = sorted_df[sorted_df["is_diabetic_friendly"] == 0].head(2).copy()
    top_unfriendly["diabetic_note"] = "not suitable"

    top_recipes = pd.concat([top_friendly, top_unfriendly])
else:
    top_recipes = df.sort_values(by="weighted_score", ascending=False).head(5).copy()
    top_recipes["diabetic_note"] = df["is_diabetic_friendly"].replace({1: "suitable", 0: "not suitable"})

print(top_recipes[["recipeName", "weighted_score", "diabetic_note"]])

                                            recipeName  weighted_score  \
360                              nut and seed biscuits              55   
642             creole spiced bean and vegetable salad              45   
71   chargrilled coriander fish with green chilli a...              40   
696                         one pot greek chicken rice              65   
698                          chicken satay asian salad              55   

    diabetic_note  
360      suitable  
642      suitable  
71       suitable  
696  not suitable  
698  not suitable  
